# Data merge and feature engineering


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import string

import re
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from tqdm.notebook import tqdm
from tqdm import tqdm
import pymorphy3
from collections import Counter

import os
from pathlib import Path
import json

In [2]:
df_weather = pd.read_csv("../data/weather_processed.csv")
df_war_events = pd.read_csv("../data/war_events_processed.csv")
df_isw = pd.read_csv("../data/isw_processed.csv")
df_isw_matrix = pd.read_csv("../data/isw_processed_svd.csv")
df_tg = pd.read_csv("../data/telegram_processed.csv")

In [3]:
df_weather['datetime_hour'] = pd.to_datetime(df_weather['datetime_hour'], errors="coerce")
df_war_events['datetime_hour'] = pd.to_datetime(df_war_events['datetime_hour'], errors="coerce")

In [4]:
df_final = pd.merge(
    df_weather, 
    df_war_events[['datetime_hour', 'region_id', 'alarm_active', 'alarm_minutes_in_hour']], 
    on=['datetime_hour', 'region_id'], 
    how='left'
)
df_final.sample(20)

,city_latitude,city_longitude,city_name,city_timezone,day_datetime,day_temp,day_dew,day_humidity,day_precip,day_precipprob,...,day_conditions_simple_Rain,day_conditions_simple_Snow,hour_conditions_simple_Clear,hour_conditions_simple_Cloudy,hour_conditions_simple_Rain,hour_conditions_simple_Snow,region_key,region_id,alarm_active,alarm_minutes_in_hour
316332,46.47250,30.73710,Odesa,Europe/Kiev,2023-08-27,27.3,18.0,60.6,0.000,0.0,...,False,False,True,False,False,False,Одеська,15,1,60.000000
524023,48.92260,24.71470,Ivano-Frankivsk,Europe/Kiev,2024-08-21,23.8,16.9,72.1,0.000,0.0,...,False,False,True,False,False,False,Івано-Франківська,9,0,0.000000
314446,51.49370,31.29440,Chernihiv,Europe/Kiev,2023-08-23,19.0,10.7,62.5,0.300,100.0,...,True,False,False,True,False,False,Чернігівська,25,0,0.000000
444551,50.45060,30.52430,Kyiv,Europe/Kiev,2024-04-05,8.8,2.7,67.8,0.000,0.0,...,False,False,False,True,False,False,Київ,26,0,0.000000
86078,50.61927,26.25131,Rivne,Europe/Kiev,2022-07-23,24.1,15.2,60.4,0.100,100.0,...,True,False,True,False,False,False,Рівненська,17,0,0.000000
35794,49.84440,24.02540,Lviv,Europe/Kiev,2022-04-27,9.6,5.0,75.0,0.000,0.0,...,False,False,False,True,False,False,Львівська,13,0,0.000000
341036,49.44070,32.06370,Cherkasy,Europe/Kiev,2023-10-09,6.8,0.2,64.1,0.500,100.0,...,True,False,True,False,False,False,Черкаська,23,0,0.000000
271464,49.23360,28.44860,Vinnytsia,Europe/Kiev,2023-06-10,20.7,11.7,59.7,0.300,100.0,...,True,False,False,True,False,False,Вінницька,2,0,0.000000
290477,48.62640,22.28510,Uzhgorod,Europe/Uzhgorod,2023-07-13,20.2,18.1,87.5,13.282,100.0,...,True,False,False,False,True,False,Закарпатська,7,0,0.000000
554937,48.50850,32.26560,Kropyvnytskyi,Europe/Kiev,2024-10-14,10.4,7.0,81.5,0.100,100.0,...,True,False,False,True,False,False,Кіровоградська,11,1,10.150000


In [5]:
df_final = df_final.sort_values(['region_id', 'datetime_hour'])
df_final.head()

,city_latitude,city_longitude,city_name,city_timezone,day_datetime,day_temp,day_dew,day_humidity,day_precip,day_precipprob,...,day_conditions_simple_Rain,day_conditions_simple_Snow,hour_conditions_simple_Clear,hour_conditions_simple_Cloudy,hour_conditions_simple_Rain,hour_conditions_simple_Snow,region_key,region_id,alarm_active,alarm_minutes_in_hour
0,49.2336,28.4486,Vinnytsia,Europe/Kiev,2022-02-24,2.8,-0.3,80.5,0.3,100.0,...,False,True,False,True,False,False,Вінницька,2,0,0.0
24,49.2336,28.4486,Vinnytsia,Europe/Kiev,2022-02-24,2.8,-0.3,80.5,0.3,100.0,...,False,True,False,True,False,False,Вінницька,2,0,0.0
48,49.2336,28.4486,Vinnytsia,Europe/Kiev,2022-02-24,2.8,-0.3,80.5,0.3,100.0,...,False,True,False,True,False,False,Вінницька,2,0,0.0
72,49.2336,28.4486,Vinnytsia,Europe/Kiev,2022-02-24,2.8,-0.3,80.5,0.3,100.0,...,False,True,False,True,False,False,Вінницька,2,0,0.0
96,49.2336,28.4486,Vinnytsia,Europe/Kiev,2022-02-24,2.8,-0.3,80.5,0.3,100.0,...,False,True,False,True,False,False,Вінницька,2,0,0.0


In [6]:
df_final.info()

<class 'pandas.DataFrame'>
Index: 634752 entries, 0 to 634751
Data columns (total 56 columns):
 #   Column                         Non-Null Count   Dtype         
---  ------                         --------------   -----         
 0   city_latitude                  634752 non-null  float64       
 1   city_longitude                 634752 non-null  float64       
 2   city_name                      634752 non-null  str           
 3   city_timezone                  634752 non-null  str           
 4   day_datetime                   634752 non-null  str           
 5   day_temp                       634752 non-null  float64       
 6   day_dew                        634752 non-null  float64       
 7   day_humidity                   634752 non-null  float64       
 8   day_precip                     634752 non-null  float64       
 9   day_precipprob                 634752 non-null  float64       
 10  day_precipcover                634752 non-null  float64       
 11  day_snow        

In [7]:
df_final['alarms_in_last_24h'] = df_final.groupby('region_id')['alarm_active'].rolling(window=24, min_periods=1).sum().reset_index(level=0, drop=True)
df_final['alarm_in_last_hour'] = df_final.groupby('region_id')['alarm_active'].shift(1)
df_final['total_active_alarms'] = df_final.groupby('datetime_hour')['alarm_active'].transform('sum')
df_final['is_weekend'] = df_final['day_of_week'].isin([5, 6]).astype(int)
df_final['is_night'] = ((df_final['hour'] >= 23) | (df_final['hour'] <= 6)).astype(int)

df_final.sample(10)

,city_latitude,city_longitude,city_name,city_timezone,day_datetime,day_temp,day_dew,day_humidity,day_precip,day_precipprob,...,hour_conditions_simple_Snow,region_key,region_id,alarm_active,alarm_minutes_in_hour,alarms_in_last_24h,alarm_in_last_hour,total_active_alarms,is_weekend,is_night
269240,50.4506,30.5243,Kyiv,Europe/Kiev,2023-06-06,19.6,7.3,48.6,0.0,0.0,...,False,Київська,10,0,0.00,5.0,0.0,1,0,0
453676,50.2536,28.6654,Zhytomyr,Europe/Kiev,2024-04-21,5.5,3.6,88.3,19.0,100.0,...,False,Житомирська,6,0,0.00,0.0,0.0,1,1,0
295888,49.5527,25.5889,Ternopil,Europe/Kiev,2023-07-22,18.9,15.0,79.2,2.5,100.0,...,False,Тернопільська,19,0,0.00,0.0,0.0,3,1,0
361783,48.9226,24.7147,Ivano-Frankivsk,Europe/Kiev,2023-11-14,7.4,3.4,76.4,3.0,100.0,...,False,Івано-Франківська,9,1,13.25,1.0,0.0,6,0,1
369899,46.9734,31.9852,Mykolaiv,Europe/Kiev,2023-11-28,2.1,-0.8,81.7,0.3,100.0,...,False,Миколаївська,14,0,0.00,6.0,1.0,3,0,1
234214,51.4937,31.2944,Chernihiv,Europe/Kiev,2023-04-06,11.8,6.2,69.7,1.0,100.0,...,False,Чернігівська,25,1,32.45,1.0,0.0,2,0,0
209619,48.0020,37.8145,Donetsk,Europe/Kiev,2023-02-22,-1.8,-6.2,72.7,1.5,100.0,...,False,Донецька,5,0,0.00,12.0,0.0,0,0,0
149746,49.8444,24.0254,Lviv,Europe/Kiev,2022-11-10,9.2,6.8,85.3,2.5,100.0,...,False,Львівська,13,0,0.00,0.0,0.0,5,0,1
406758,47.8289,35.1626,Zaporozhye,Europe/Zaporozhye,2024-01-31,0.6,-1.6,85.2,0.0,0.0,...,False,Запорізька,8,0,0.00,10.0,0.0,0,0,1
182695,48.9226,24.7147,Ivano-Frankivsk,Europe/Kiev,2023-01-07,2.1,0.9,92.1,0.9,100.0,...,False,Івано-Франківська,9,0,0.00,3.0,0.0,0,1,1


In [8]:
"""neighbouring_regions = {
    1: [21],
    2: [6, 10, 11, 15, 22, 23, 24],
    3: [13, 17],
    4: [5, 8, 11, 14, 16, 20, 21],
    5: [4, 8, 12, 20],
    6: [2, 10, 17, 22],
    7: [9, 13],
    8: [4, 5, 21],
    9: [7, 13, 19, 24],
    10: [2, 6, 16, 23, 25],
    11: [2, 4, 14, 15, 16, 23],
    12: [5, 20],
    13: [3, 7, 9, 17, 19],
    14: [4, 11, 15, 21],
    15: [2, 11, 14],
    16: [4, 10, 11, 18, 20, 23, 25],
    17: [3, 6, 13, 19, 22],
    18: [16, 20, 25],
    19: [9, 13, 17, 22, 24],
    20: [4, 5, 12, 16, 18],
    21: [1, 4, 8, 14],
    22: [2, 6, 17, 19, 24],
    23: [2, 10, 11, 16],
    24: [2, 9, 19, 22],
    25: [10, 16, 18], 
    26: [10]
}

alarms_matrix = df_final.pivot_table(index='datetime_hour', columns='region_id', values='alarm_active', fill_value=0)

def get_neighbour_sum_safe(row):
    region_id = row['region_id']
    datetime = row['datetime_hour']
    
    if region_id in neighbouring_regions:
        neighbours = neighbouring_regions[region_id]
        
        if datetime in alarms_matrix.index:
            valid_neighbours = [n for n in neighbours if n in alarms_matrix.columns]

            if valid_neighbours:
                return alarms_matrix.loc[datetime, valid_neighbours].sum()
    return 0

df_final['neighbour_alarms'] = df_final.apply(get_neighbour_sum_safe, axis=1)

df_final.sample(15)"""

"neighbouring_regions = {\n    1: [21],\n    2: [6, 10, 11, 15, 22, 23, 24],\n    3: [13, 17],\n    4: [5, 8, 11, 14, 16, 20, 21],\n    5: [4, 8, 12, 20],\n    6: [2, 10, 17, 22],\n    7: [9, 13],\n    8: [4, 5, 21],\n    9: [7, 13, 19, 24],\n    10: [2, 6, 16, 23, 25],\n    11: [2, 4, 14, 15, 16, 23],\n    12: [5, 20],\n    13: [3, 7, 9, 17, 19],\n    14: [4, 11, 15, 21],\n    15: [2, 11, 14],\n    16: [4, 10, 11, 18, 20, 23, 25],\n    17: [3, 6, 13, 19, 22],\n    18: [16, 20, 25],\n    19: [9, 13, 17, 22, 24],\n    20: [4, 5, 12, 16, 18],\n    21: [1, 4, 8, 14],\n    22: [2, 6, 17, 19, 24],\n    23: [2, 10, 11, 16],\n    24: [2, 9, 19, 22],\n    25: [10, 16, 18], \n    26: [10]\n}\n\nalarms_matrix = df_final.pivot_table(index='datetime_hour', columns='region_id', values='alarm_active', fill_value=0)\n\ndef get_neighbour_sum_safe(row):\n    region_id = row['region_id']\n    datetime = row['datetime_hour']\n\n    if region_id in neighbouring_regions:\n        neighbours = neighbouring_

In [9]:
def calculate_hours_since_last(series):
    groups = series.cumsum()
    return series.groupby(groups).cumcount()

df_final['hours_since_last_alarm'] = df_final.groupby('region_id')['alarm_active'].transform(calculate_hours_since_last)

In [10]:
"""df_final['neighbour_alarms'] = df_final['neighbour_alarms'].astype(int)
df_final['alarms_in_last_24h'] = df_final['alarms_in_last_24h'].astype(int)
df_final['alarm_in_last_hour'] = df_final['alarm_in_last_hour'].fillna(0).astype(int)"""

"df_final['neighbour_alarms'] = df_final['neighbour_alarms'].astype(int)\ndf_final['alarms_in_last_24h'] = df_final['alarms_in_last_24h'].astype(int)\ndf_final['alarm_in_last_hour'] = df_final['alarm_in_last_hour'].fillna(0).astype(int)"

In [ ]:
df_final["alarm_lag_1"] = df_final.groupby("region_id")["alarm_active"].shift(1)
df_final["alarm_lag_3"] = df_final.groupby("region_id")["alarm_active"].shift(3)
df_final["alarm_lag_6"] = df_final.groupby("region_id")["alarm_active"].shift(6)
df_final["alarm_lag_12"] = df_final.groupby("region_id")["alarm_active"].shift(12)

In [11]:
df_final.sample(20)

,city_latitude,city_longitude,city_name,city_timezone,day_datetime,day_temp,day_dew,day_humidity,day_precip,day_precipprob,...,region_key,region_id,alarm_active,alarm_minutes_in_hour,alarms_in_last_24h,alarm_in_last_hour,total_active_alarms,is_weekend,is_night,hours_since_last_alarm
348365,48.62640,22.28510,Uzhgorod,Europe/Uzhgorod,2023-10-21,18.2,12.4,69.2,0.093,100.0,...,Закарпатська,7,0,0.0,0.0,0.0,1,1,0,529
59650,49.84440,24.02540,Lviv,Europe/Kiev,2022-06-07,21.0,11.0,55.1,0.000,0.0,...,Львівська,13,0,0.0,0.0,0.0,2,0,0,55
374484,46.47250,30.73710,Odesa,Europe/Kiev,2023-12-06,1.3,-0.8,85.6,0.000,0.0,...,Одеська,15,0,0.0,3.0,0.0,1,0,1,5
597701,48.62640,22.28510,Uzhgorod,Europe/Uzhgorod,2024-12-27,1.3,-4.5,68.2,0.000,0.0,...,Закарпатська,7,0,0.0,0.0,0.0,6,0,0,54
512274,46.64010,32.61420,Kherson,Europe/Kiev,2024-08-01,24.3,10.1,43.6,0.000,0.0,...,Херсонська,21,0,0.0,11.0,0.0,5,0,0,7
386900,49.44070,32.06370,Cherkasy,Europe/Kiev,2023-12-27,5.8,1.0,71.5,0.600,100.0,...,Черкаська,23,0,0.0,7.0,0.0,2,0,0,4
232079,50.45060,30.52430,Kyiv,Europe/Kiev,2023-04-02,8.2,7.6,95.7,8.000,100.0,...,Київ,26,0,0.0,0.0,0.0,3,1,0,70
598185,48.50850,32.26560,Kropyvnytskyi,Europe/Kiev,2024-12-28,-1.1,-2.5,90.1,0.200,100.0,...,Кіровоградська,11,0,0.0,0.0,0.0,1,1,0,37
115593,48.50850,32.26560,Kropyvnytskyi,Europe/Kiev,2022-09-12,13.6,12.2,92.0,3.500,100.0,...,Кіровоградська,11,0,0.0,4.0,0.0,6,0,0,12
107459,46.97336,31.98522,Mykolaiv,Europe/Kiev,2022-08-29,26.5,18.2,62.5,0.000,0.0,...,Миколаївська,14,0,0.0,7.0,0.0,0,0,0,2


In [12]:
df_isw_matrix.sample(10)

,date,isw_topic_0,isw_topic_1,isw_topic_2,isw_topic_3,isw_topic_4,isw_topic_5,isw_topic_6,isw_topic_7,isw_topic_8,...,isw_topic_140,isw_topic_141,isw_topic_142,isw_topic_143,isw_topic_144,isw_topic_145,isw_topic_146,isw_topic_147,isw_topic_148,isw_topic_149
398,2023-03-31,0.761592,-0.224080,-0.103312,0.108694,0.034220,-0.061863,-0.051728,0.160928,-0.003087,...,-0.048283,-0.012349,-0.015349,-0.003089,0.006748,-0.021881,-0.018195,-0.021508,0.005232,-0.019257
711,2024-02-10,0.788466,-0.056842,-0.230045,0.130822,0.028524,-0.018018,0.006671,0.021234,0.023288,...,-0.018380,-0.006045,0.001810,-0.013014,-0.001454,-0.014119,0.012556,0.019708,-0.019591,-0.020591
796,2024-05-05,0.834839,0.019762,-0.088477,0.101592,-0.042838,-0.005648,0.005062,-0.050947,0.041944,...,0.021944,0.015333,0.009553,0.030924,-0.006878,0.020426,-0.000862,0.015565,0.010335,0.021487
1036,2025-01-04,0.770646,0.187815,-0.048848,0.028963,0.220483,0.174501,-0.109238,-0.175354,0.145724,...,0.001642,-0.014533,-0.009684,0.006698,-0.024724,0.009841,-0.002471,0.003002,0.016194,-0.027039
188,2022-08-30,0.775517,-0.239928,0.170380,-0.106069,-0.062923,0.085875,0.006087,0.075233,0.133559,...,0.023371,0.006030,0.025769,0.034940,-0.030922,0.005004,-0.020017,0.012893,-0.002795,-0.015808
31,2022-03-26,0.670213,-0.175590,0.239837,0.188369,0.092676,-0.132588,-0.057576,0.095722,0.090243,...,0.001279,-0.021295,-0.006134,-0.016880,-0.002076,-0.032829,-0.001868,-0.002118,-0.040842,0.002264
144,2022-07-17,0.718017,-0.220815,0.252538,-0.009127,-0.081794,0.139607,0.081364,-0.116829,-0.009575,...,0.009367,-0.018705,-0.066173,0.014977,-0.007061,0.007710,0.034196,0.016834,-0.020832,0.020157
477,2023-06-18,0.802408,-0.176948,-0.019841,-0.073563,-0.078515,0.068766,0.026135,-0.014657,-0.084663,...,-0.012437,-0.017019,0.006043,-0.008466,-0.001430,0.004210,0.007942,-0.011742,-0.038251,-0.015525
455,2023-05-27,0.786571,-0.236401,-0.060035,-0.005343,-0.003228,0.040632,-0.008143,0.060293,-0.086701,...,0.014437,-0.024329,0.010762,-0.041915,0.020630,-0.030132,0.002301,0.002438,0.007609,0.006869
255,2022-11-05,0.792842,-0.212619,-0.000934,-0.180506,0.066913,-0.108634,0.066170,-0.065000,0.077111,...,0.017400,-0.025180,-0.016517,0.006636,0.007238,-0.007408,-0.010627,0.009397,0.032664,-0.011384


In [13]:
df_isw_matrix["date"] = pd.to_datetime(df_isw_matrix["date"]) + pd.Timedelta(days=1)
df_isw_matrix = df_isw_matrix.rename(columns={'date': 'day_datetime'})

In [14]:
df_final['day_datetime'] = pd.to_datetime(df_final['day_datetime']).dt.date
df_isw_matrix['day_datetime'] = pd.to_datetime(df_isw_matrix['day_datetime']).dt.date
df_isw_matrix = df_isw_matrix.fillna(0, inplace=True)
df_isw_matrix.isna().sum()

day_datetime     0
isw_topic_0      0
isw_topic_1      0
isw_topic_2      0
isw_topic_3      0
                ..
isw_topic_145    0
isw_topic_146    0
isw_topic_147    0
isw_topic_148    0
isw_topic_149    0
Length: 151, dtype: int64

In [15]:
df_final = df_final.merge(df_isw_matrix, on="day_datetime", how="left")

In [23]:
df_final.sample(10)

,city_latitude,city_longitude,city_name,city_timezone,day_datetime,day_temp,day_dew,day_humidity,day_precip,day_precipprob,...,isw_topic_140,isw_topic_141,isw_topic_142,isw_topic_143,isw_topic_144,isw_topic_145,isw_topic_146,isw_topic_147,isw_topic_148,isw_topic_149
440594,49.55270,25.58890,Ternopil,Europe/Kiev,2024-02-03,3.9,1.8,86.2,2.000,100.0,...,0.036308,-0.016159,-0.005081,0.027757,0.028221,-0.043939,0.000272,0.000782,-0.014494,0.016103
40986,50.74690,25.32630,Lutsk,Europe/Kiev,2023-10-20,11.4,10.6,94.7,10.451,100.0,...,-0.007727,-0.023455,-0.004080,-0.000604,-0.032064,-0.028032,0.015496,-0.003552,0.007470,-0.018506
33176,50.74690,25.32630,Lutsk,Europe/Kiev,2022-11-29,0.5,-0.7,91.7,0.100,100.0,...,-0.008931,-0.012027,0.005855,-0.010283,0.002359,-0.021880,-0.005987,0.031007,-0.006387,-0.000966
444490,49.55270,25.58890,Ternopil,Europe/Kiev,2024-07-14,27.6,18.5,59.2,2.400,100.0,...,0.000187,0.006503,0.007955,-0.027808,-0.003866,0.009190,-0.010353,-0.035398,-0.003460,-0.007162
293064,46.97336,31.98522,Mykolaiv,Europe/Kiev,2022-05-12,15.2,8.0,63.4,0.000,0.0,...,-0.056699,0.029085,-0.004526,-0.012452,0.006574,0.028420,0.018159,-0.000168,-0.015358,-0.008218
114726,50.25360,28.66540,Zhytomyr,Europe/Kiev,2023-02-26,2.5,-0.5,81.0,0.100,100.0,...,-0.013662,0.034461,0.002031,-0.010658,-0.024291,0.015042,0.010664,-0.006611,0.013999,0.021826
39632,50.74690,25.32630,Lutsk,Europe/Kiev,2023-08-25,19.7,13.9,72.1,0.200,100.0,...,0.018742,0.028117,-0.015506,0.049666,0.002416,-0.020899,0.017747,-0.002169,0.006511,-0.007144
534952,49.44070,32.06370,Cherkasy,Europe/Kiev,2022-10-10,11.0,7.1,77.7,0.300,100.0,...,-0.031410,-0.014775,0.001172,0.026616,-0.012282,-0.002694,0.000053,-0.033567,-0.023506,-0.039006
232965,50.45060,30.52430,Kyiv,Europe/Kiev,2024-07-24,22.3,16.2,70.4,0.700,100.0,...,-0.023535,0.008379,-0.015938,-0.002074,-0.026821,-0.010418,0.016981,0.018085,0.021895,0.011565
504360,49.41680,26.97430,Khmelnytskyi,Europe/Kiev,2022-04-22,8.5,4.8,79.1,6.000,100.0,...,0.017509,-0.019684,0.010469,0.022138,0.037293,-0.068848,0.004278,-0.047774,0.017952,0.021319


In [17]:
df_final.isna().sum()

city_latitude        0
city_longitude       0
city_name            0
city_timezone        0
day_datetime         0
                  ... 
isw_topic_145     6336
isw_topic_146     6336
isw_topic_147     6336
isw_topic_148     6336
isw_topic_149     6336
Length: 212, dtype: int64

In [21]:
df_final.fillna(0, inplace=True)

,city_latitude,city_longitude,city_name,city_timezone,day_datetime,day_temp,day_dew,day_humidity,day_precip,day_precipprob,...,isw_topic_140,isw_topic_141,isw_topic_142,isw_topic_143,isw_topic_144,isw_topic_145,isw_topic_146,isw_topic_147,isw_topic_148,isw_topic_149
0,49.2336,28.4486,Vinnytsia,Europe/Kiev,2022-02-24,2.8,-0.3,80.5,0.3,100.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,49.2336,28.4486,Vinnytsia,Europe/Kiev,2022-02-24,2.8,-0.3,80.5,0.3,100.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,49.2336,28.4486,Vinnytsia,Europe/Kiev,2022-02-24,2.8,-0.3,80.5,0.3,100.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,49.2336,28.4486,Vinnytsia,Europe/Kiev,2022-02-24,2.8,-0.3,80.5,0.3,100.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,49.2336,28.4486,Vinnytsia,Europe/Kiev,2022-02-24,2.8,-0.3,80.5,0.3,100.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
635323,50.4506,30.5243,Kyiv,Europe/Kiev,2025-03-01,0.0,-2.6,82.9,0.7,100.0,...,-0.004258,0.016078,0.006146,0.014859,0.000348,-0.002017,0.002455,-0.008404,-0.019102,0.004664
635324,50.4506,30.5243,Kyiv,Europe/Kiev,2025-03-01,0.0,-2.6,82.9,0.7,100.0,...,-0.004258,0.016078,0.006146,0.014859,0.000348,-0.002017,0.002455,-0.008404,-0.019102,0.004664
635325,50.4506,30.5243,Kyiv,Europe/Kiev,2025-03-01,0.0,-2.6,82.9,0.7,100.0,...,-0.004258,0.016078,0.006146,0.014859,0.000348,-0.002017,0.002455,-0.008404,-0.019102,0.004664
635326,50.4506,30.5243,Kyiv,Europe/Kiev,2025-03-01,0.0,-2.6,82.9,0.7,100.0,...,-0.004258,0.016078,0.006146,0.014859,0.000348,-0.002017,0.002455,-0.008404,-0.019102,0.004664


In [22]:
df_final.isna().sum()

city_latitude     0
city_longitude    0
city_name         0
city_timezone     0
day_datetime      0
                 ..
isw_topic_145     0
isw_topic_146     0
isw_topic_147     0
isw_topic_148     0
isw_topic_149     0
Length: 212, dtype: int64

### Feature engineering for isw topics 

In [27]:
topic_cols = [c for c in df_final.columns if "isw_topic_" in c]
df_isw_abs = df_final[topic_cols].abs()

df_final['isw_total_intensity'] = df_isw_abs.sum(axis=1)
df_final['isw_topic_std'] = df_isw_abs.std(axis=1)
df_final['isw_topic_max'] = df_isw_abs.max(axis=1)
df_final["isw_topic_mean"] = df_isw_abs.mean(axis=1)

df_final['isw_velocity_24h'] = df_final['isw_total_intensity'].diff(24)

df_final["isw_intensity_ema"] = (df_final.groupby("region_id")["isw_total_intensity"].transform(lambda x: x.ewm(span=24).mean()))

df_final["isw_topic_entropy"] = -(df_isw_abs.div(df_isw_abs.sum(axis=1), axis=0) * np.log(df_isw_abs.div(df_isw_abs.sum(axis=1), axis=0) + 1e-9)).sum(axis=1)

"""df_final.drop(columns=topic_cols, inplace=True)
df_final = df_final.copy()"""
df_final.fillna(0, inplace=True)

,city_latitude,city_longitude,city_name,city_timezone,day_datetime,day_temp,day_dew,day_humidity,day_precip,day_precipprob,...,isw_topic_145,isw_topic_146,isw_topic_147,isw_topic_148,isw_topic_149,isw_total_intensity,isw_intensity_24h_avg,isw_velocity_24h,isw_intensity_7d_avg,isw_velocity_7d
0,49.2336,28.4486,Vinnytsia,Europe/Kiev,2022-02-24,2.8,-0.3,80.5,0.3,100.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,49.2336,28.4486,Vinnytsia,Europe/Kiev,2022-02-24,2.8,-0.3,80.5,0.3,100.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,49.2336,28.4486,Vinnytsia,Europe/Kiev,2022-02-24,2.8,-0.3,80.5,0.3,100.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,49.2336,28.4486,Vinnytsia,Europe/Kiev,2022-02-24,2.8,-0.3,80.5,0.3,100.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,49.2336,28.4486,Vinnytsia,Europe/Kiev,2022-02-24,2.8,-0.3,80.5,0.3,100.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
635323,50.4506,30.5243,Kyiv,Europe/Kiev,2025-03-01,0.0,-2.6,82.9,0.7,100.0,...,-0.002017,0.002455,-0.008404,-0.019102,0.004664,1.367157,1.369791,-0.015808,1.299188,0.124945
635324,50.4506,30.5243,Kyiv,Europe/Kiev,2025-03-01,0.0,-2.6,82.9,0.7,100.0,...,-0.002017,0.002455,-0.008404,-0.019102,0.004664,1.367157,1.369133,-0.015808,1.299932,0.124945
635325,50.4506,30.5243,Kyiv,Europe/Kiev,2025-03-01,0.0,-2.6,82.9,0.7,100.0,...,-0.002017,0.002455,-0.008404,-0.019102,0.004664,1.367157,1.368474,-0.015808,1.300675,0.124945
635326,50.4506,30.5243,Kyiv,Europe/Kiev,2025-03-01,0.0,-2.6,82.9,0.7,100.0,...,-0.002017,0.002455,-0.008404,-0.019102,0.004664,1.367157,1.367815,-0.015808,1.301419,0.124945


### TELEGRAM MERGE